# 1) MLP: A * B % C (A,B between 0..99, C 0..19)

In [1]:
#@title (0) Setup
from google.colab import drive
import sys
drive.mount('/content/drive')
#sys.path.append('/content/drive/MyDrive/Colab Notebooks')

%run "/content/drive/MyDrive/Colab Notebooks/Build your own LLM/han-common.ipynb"

Mounted at /content/drive
Train: 152,000 samples  |  Test: 38,000 samples
Input dim: 219  |  Output classes: 19
Label distribution (train): {np.int64(0): np.int64(51411), np.int64(1): np.int64(11977), np.int64(2): np.int64(14363), np.int64(3): np.int64(10413), np.int64(4): np.int64(11615), np.int64(5): np.int64(6717), np.int64(6): np.int64(10053), np.int64(7): np.int64(4823), np.int64(8): np.int64(6869), np.int64(9): np.int64(4650), np.int64(10): np.int64(4769), np.int64(11): np.int64(2542), np.int64(12): np.int64(4452), np.int64(13): np.int64(1747), np.int64(14): np.int64(2023), np.int64(15): np.int64(1374), np.int64(16): np.int64(1254), np.int64(17): np.int64(538), np.int64(18): np.int64(410)}
DNN(
  (proj_in): Linear(in_features=219, out_features=128, bias=True)
  (layers): ModuleList(
    (0-15): 16 x Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
    )
  )
  (norms): Modu

## 1.1) Dataset

In [2]:
# ── Label noise ──
NOISE_RATE = 0.1
NUM_CLASSES = int(y_all.max()) + 1

noisy_idx = np.random.choice(len(y_train), size=int(NOISE_RATE * len(y_train)), replace=False)
for i in noisy_idx:
    wrong_labels = [l for l in range(NUM_CLASSES) if l != y_train[i]]
    y_train[i] = np.random.choice(wrong_labels)

# DataLoaders
train_ds = TensorDataset(torch.tensor(X_train).to(DEVICE), torch.tensor(y_train, dtype=torch.long).to(DEVICE))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print(f"Input dim: {X_train.shape[1]}  |  Output classes: {NUM_CLASSES}")
print(f"Label noise: {len(noisy_idx):,} / {len(y_train):,} = {NOISE_RATE:.0%} of training labels corrupted")

Input dim: 219  |  Output classes: 19
Label noise: 15,200 / 152,000 = 10% of training labels corrupted


## 1.3) Train

In [3]:
def train(run_name="", weight_decay=0.0, clip_grad=False):
    model = DNN().to(DEVICE)
    model = train_common(model, run_name, weight_decay_c=weight_decay, clip_grad_c=clip_grad)

    history = GetHistory(run_name)
    if history is not None:
        visualize_individual_run(history, run_name)
    else:
        print(f"Warning: Could not find history for run '{run_name}' in all_runs.")

    return model

# 2) Weight decay 0

In [ ]:
MODEL = train(run_name="weight decay 0", weight_decay=0.0)

  🚀 Run: weight decay 0
Epoch      1 | Train loss 1.9239  acc 41.5% | Test loss 1.6414  acc 44.5%
Epoch      2 | Train loss 1.6174  acc 48.9% | Test loss 1.3161  acc 52.8%
Epoch      3 | Train loss 1.5019  acc 50.0% | Test loss 1.1967  acc 53.8%
Epoch      4 | Train loss 1.4926  acc 50.0% | Test loss 1.1991  acc 54.1%
Epoch      5 | Train loss 1.4751  acc 50.7% | Test loss 1.1898  acc 54.0%
Epoch      6 | Train loss 1.3807  acc 54.9% | Test loss 1.0757  acc 58.7%
Epoch      7 | Train loss 1.3540  acc 55.9% | Test loss 1.0459  acc 59.9%
Epoch      8 | Train loss 1.3356  acc 56.4% | Test loss 1.0388  acc 60.0%
Epoch      9 | Train loss 1.3048  acc 58.1% | Test loss 1.0165  acc 61.8%
Epoch     10 | Train loss 1.2485  acc 61.2% | Test loss 0.9666  acc 64.9%
Epoch     11 | Train loss 1.1539  acc 65.4% | Test loss 0.8801  acc 69.9%
Epoch     12 | Train loss 1.1179  acc 66.1% | Test loss 0.8572  acc 70.5%
Epoch     13 | Train loss 1.0902  acc 67.6% | Test loss 0.8437  acc 72.1%
Epoch     14 |

# 3) Weight decay 0.0001

In [ ]:
MODEL = train(run_name="weight decay 0.0001", weight_decay=0.0001)

  🚀 Run: weight decay 0.0001
Epoch      1 | Train loss 1.9175  acc 41.7% | Test loss 1.6349  acc 44.9%
Epoch      2 | Train loss 1.6096  acc 49.2% | Test loss 1.3155  acc 53.3%
Epoch      3 | Train loss 1.5063  acc 49.9% | Test loss 1.1946  acc 53.8%
Epoch      4 | Train loss 1.4976  acc 50.0% | Test loss 1.2009  acc 54.1%
Epoch      5 | Train loss 1.4817  acc 50.4% | Test loss 1.1981  acc 53.8%
Epoch      6 | Train loss 1.4206  acc 53.6% | Test loss 1.1261  acc 56.8%
Epoch      7 | Train loss 1.3554  acc 55.8% | Test loss 1.0523  acc 59.8%
Epoch      8 | Train loss 1.3347  acc 56.8% | Test loss 1.0328  acc 60.5%
Epoch      9 | Train loss 1.2693  acc 60.3% | Test loss 0.9780  acc 64.4%
Epoch     10 | Train loss 1.1761  acc 64.9% | Test loss 0.8626  acc 70.2%
Epoch     11 | Train loss 1.1444  acc 65.6% | Test loss 0.8531  acc 70.6%
Epoch     12 | Train loss 1.1286  acc 65.6% | Test loss 0.8668  acc 70.4%
Epoch     13 | Train loss 1.1024  acc 66.5% | Test loss 0.8640  acc 70.8%
Epoch    

# 4) Weight decay 0.001

In [ ]:
MODEL = train(run_name="weight decay 0.001", weight_decay=0.001)

  🚀 Run: weight decay 0.001
Epoch      1 | Train loss 1.9151  acc 41.7% | Test loss 1.6320  acc 44.9%
Epoch      2 | Train loss 1.6050  acc 49.3% | Test loss 1.3020  acc 53.1%
Epoch      3 | Train loss 1.4927  acc 50.1% | Test loss 1.1854  acc 54.0%
Epoch      4 | Train loss 1.4888  acc 50.2% | Test loss 1.1947  acc 54.0%
Epoch      5 | Train loss 1.4780  acc 50.8% | Test loss 1.1973  acc 54.3%
Epoch      6 | Train loss 1.3751  acc 54.8% | Test loss 1.0707  acc 58.6%
Epoch      7 | Train loss 1.3633  acc 55.7% | Test loss 1.0634  acc 60.0%
Epoch      8 | Train loss 1.3340  acc 57.3% | Test loss 1.0319  acc 60.9%
Epoch      9 | Train loss 1.2904  acc 58.3% | Test loss 1.0062  acc 62.2%
Epoch     10 | Train loss 1.2471  acc 60.9% | Test loss 0.9646  acc 65.2%
Epoch     11 | Train loss 1.1643  acc 64.9% | Test loss 0.8834  acc 69.7%
Epoch     12 | Train loss 1.1188  acc 66.1% | Test loss 0.8525  acc 70.4%
Epoch     13 | Train loss 1.1116  acc 66.2% | Test loss 0.8856  acc 69.9%
Epoch     

# 5) Weight decay 0.01

In [ ]:
MODEL = train(run_name="weight decay 0.01", weight_decay=0.01)

  🚀 Run: weight decay 0.01
Epoch      1 | Train loss 1.9234  acc 41.4% | Test loss 1.6336  acc 44.7%
Epoch      2 | Train loss 1.5977  acc 49.3% | Test loss 1.3066  acc 53.4%
Epoch      3 | Train loss 1.4957  acc 50.1% | Test loss 1.1915  acc 53.8%
Epoch      4 | Train loss 1.4870  acc 50.1% | Test loss 1.1845  acc 54.2%
Epoch      5 | Train loss 1.4650  acc 51.0% | Test loss 1.1867  acc 54.3%
Epoch      6 | Train loss 1.3733  acc 55.3% | Test loss 1.0609  acc 59.0%
Epoch      7 | Train loss 1.3554  acc 55.8% | Test loss 1.0494  acc 59.8%
Epoch      8 | Train loss 1.3386  acc 56.1% | Test loss 1.0472  acc 59.8%
Epoch      9 | Train loss 1.3224  acc 56.6% | Test loss 1.0436  acc 60.0%
Epoch     10 | Train loss 1.2869  acc 58.5% | Test loss 1.0242  acc 62.2%
Epoch     11 | Train loss 1.1932  acc 63.5% | Test loss 0.9042  acc 68.0%
Epoch     12 | Train loss 1.1282  acc 65.7% | Test loss 0.8553  acc 70.4%
Epoch     13 | Train loss 1.1025  acc 66.4% | Test loss 0.8508  acc 70.7%
Epoch     1

# 6) Weight decay 0.1

In [ ]:
MODEL = train(run_name="weight decay 0.1", weight_decay=0.1)

  🚀 Run: weight decay 0.1
Epoch      1 | Train loss 1.8747  acc 42.3% | Test loss 1.6039  acc 45.6%
Epoch      2 | Train loss 1.5514  acc 49.6% | Test loss 1.2378  acc 53.6%
Epoch      3 | Train loss 1.4860  acc 50.2% | Test loss 1.1842  acc 53.6%
Epoch      4 | Train loss 1.4738  acc 49.9% | Test loss 1.1706  acc 54.1%
Epoch      5 | Train loss 1.4390  acc 51.9% | Test loss 1.1281  acc 56.1%
Epoch      6 | Train loss 1.3557  acc 55.7% | Test loss 1.0442  acc 59.8%
Epoch      7 | Train loss 1.3461  acc 55.6% | Test loss 1.0282  acc 60.1%
Epoch      8 | Train loss 1.3373  acc 55.8% | Test loss 1.0126  acc 59.9%
Epoch      9 | Train loss 1.3350  acc 56.0% | Test loss 1.0086  acc 60.2%
Epoch     10 | Train loss 1.3244  acc 56.8% | Test loss 1.0233  acc 60.3%
Epoch     11 | Train loss 1.2820  acc 59.1% | Test loss 0.9724  acc 63.4%
Epoch     12 | Train loss 1.1711  acc 65.4% | Test loss 0.8259  acc 70.7%
Epoch     13 | Train loss 1.1266  acc 66.8% | Test loss 0.7909  acc 72.5%
Epoch     14

# 7) Weight Decay 0.1 + Gradient clipping

In [4]:
MODEL = train(run_name="Weight decay 0.1 + clipping",
      weight_decay=0.1,
      clip_grad=True)

  🚀 Run: Weight decay 0.1 + clipping
Epoch      1 | Train loss 1.5299  acc 49.2% | Test loss 1.1726  acc 53.9%
Epoch      2 | Train loss 1.4800  acc 49.6% | Test loss 1.1632  acc 54.0%
Epoch      3 | Train loss 1.4714  acc 49.6% | Test loss 1.1456  acc 53.9%
Epoch      4 | Train loss 1.4612  acc 49.8% | Test loss 1.1250  acc 54.4%
Epoch      5 | Train loss 1.3738  acc 54.9% | Test loss 1.0242  acc 60.1%
Epoch      6 | Train loss 1.3620  acc 55.2% | Test loss 1.0354  acc 60.2%
Epoch      7 | Train loss 1.3626  acc 55.1% | Test loss 1.0106  acc 60.4%
Epoch      8 | Train loss 1.3568  acc 55.2% | Test loss 1.0091  acc 60.3%
Epoch      9 | Train loss 1.3575  acc 56.4% | Test loss 0.9898  acc 61.9%
Epoch     10 | Train loss 1.3133  acc 57.9% | Test loss 0.9747  acc 63.2%
Epoch     11 | Train loss 1.2932  acc 59.5% | Test loss 0.9448  acc 64.8%
Epoch     12 | Train loss 1.1775  acc 66.3% | Test loss 0.8100  acc 72.5%
Epoch     13 | Train loss 1.1330  acc 68.2% | Test loss 0.7499  acc 75.3%
E

NameError: name 'GetHistory' is not defined

# 8) Loss + Accuracy across runs

In [ ]:
#@title Visualize
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import logging
import matplotlib.ticker as ticker

# Suppress missing font warnings
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# Prefer the deployment font when available, otherwise fall back to common sans-serif fonts
font_candidates = ['Inter', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
plt.rcParams['font.family'] = font_candidates
font_path = 'fonts/Inter-Regular.ttf'
try:
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
except Exception:
    pass

# Set styling for all text and lines
plt.rcParams['text.color'] = '#F1F5F9'
plt.rcParams['axes.labelcolor'] = '#F1F5F9'
plt.rcParams['xtick.color'] = '#F1F5F9'
plt.rcParams['ytick.color'] = '#F1F5F9'
plt.rcParams['axes.edgecolor'] = '#F1F5F9'

# Grid styling
plt.rcParams['grid.color'] = '#5B7268'
plt.rcParams['grid.alpha'] = 0.7

def _coplot_luminance_from_rgba(rgba):
    r, g, b = mcolors.to_rgb(rgba)
    def _linear(c):
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * _linear(r) + 0.7152 * _linear(g) + 0.0722 * _linear(b)

def _coplot_text_color_for_rgba(rgba, preferred='#F1F5F9'):
    try:
        preferred_l = _coplot_luminance_from_rgba(mcolors.to_rgba(preferred))
        bg_l = _coplot_luminance_from_rgba(rgba)
        lighter = max(preferred_l, bg_l)
        darker = min(preferred_l, bg_l)
        ratio = (lighter + 0.05) / (darker + 0.05)
        if ratio >= 4.5:
            return preferred
    except Exception:
        pass
    bg_l = _coplot_luminance_from_rgba(rgba)
    return '#111111' if bg_l > 0.5 else '#FFFFFF'

# Create figure and axis with publication-quality settings
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
fig.patch.set_facecolor('#1A3A2A')

# Distinct bright colors for dark background
colors = ["#38BDF8", "#34D399", "#FBBF24", "#F87171", "#A78BFA", "#FB923C", "#E879F9", "#2DD4BF"]
line_styles = ["-","--",":","-."]*2
marker_styles = ["o","s","^","D"]*2

for idx, run in enumerate(all_runs):
    h = run["history"]

    # Plot accuracy
    axes[0].plot(h["epoch"], h["test_acc"], label=run["name"],
                 color=colors[idx % len(colors)], linewidth=2, alpha=0.9)

    # Plot loss
    axes[1].plot(h["epoch"], h["test_loss"], label=run["name"],
                 color=colors[idx % len(colors)], linewidth=2, alpha=0.9)

# Apply custom styling to both axes
for ax in axes:
    ax.set_facecolor('#1A3A2A')
    ax.set_xlabel('Epoch', fontsize=12, labelpad=10, color='#F1F5F9')

    # Force integer epochs
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    # Add grid
    ax.grid(axis='both', alpha=0.7, color='#5B7268', linestyle='--', linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)

    # Remove top and right spines for cleaner look
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_color('#F1F5F9')
    ax.spines['left'].set_color('#F1F5F9')

    # Adjust spine positions
    ax.spines['left'].set_position(('outward', 2))
    ax.spines['bottom'].set_position(('outward', 2))

    # Add legend
    ax.legend(frameon=False, fontsize=10)

# Specific titles and labels
axes[0].set_ylabel('Test Accuracy', fontsize=12, color='#F1F5F9', labelpad=10)
axes[0].set_title('Test Accuracy', color='#F1F5F9', pad=15)
axes[0].yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

axes[1].set_ylabel('Test Loss', fontsize=12, color='#F1F5F9', labelpad=10)
axes[1].set_title('Test Loss', color='#F1F5F9', pad=15)
axes[1].set_yscale('linear')
axes[1].set_ylim(bottom=0.0)
axes[1].yaxis.set_major_formatter(ticker.ScalarFormatter())

# Global title
plt.suptitle("Weight Decay Comparison", fontsize=14, y=1.02, color='#F1F5F9')

# Tight layout
plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Run':<30s} {'Converged Epoch':<20s} {'Final Test Acc':<15s}")
print("-" * 65)
for run in sorted(all_runs, key=lambda x: (x['converged_epoch'] is None, x['converged_epoch'])):
    ep = run["converged_epoch"]
    final_acc = run["history"]["test_acc"][-1]
    print(f"{run['name']:<30s} {str(ep) if ep else 'Did not converge':<20s} {final_acc:.1%}")


Run                            Converged Epoch      Final Test Acc 
-----------------------------------------------------------------
weight decay 0                 Did not converge     81.3%
weight decay 0.0001            Did not converge     80.5%
weight decay 0.001             Did not converge     79.7%
weight decay 0.01              Did not converge     84.3%
weight decay 0.1               Did not converge     93.3%
Weight decay 0.1 + clipping    Did not converge     92.4%


In [5]:
#@title (9) Save and load model
FILENAME = "DDN-v6.pth"
MODEL = get_model('Weight decay 0.1 + clipping')

print("--- Testing Local Save & Load ---")
load_model = DNN().to(DEVICE)
load_model = save_load_model_local(MODEL, load_model, filename_c=FILENAME)
predicted_value = invoke_model_with_a_b_c(load_model, 18, 1, 10)

--- Testing Local Save & Load ---
model weights saved locally to: .//DDN-v6.pth
model weights loaded successfully into a new model instance.
Prediction for (18 * 1) % 10 = 8


In [6]:
#@title (10) Save to Google Drive
pathname = save_to_google_drive(MODEL, FILENAME)

Model saved to Google Drive: /content/drive/MyDrive/workshop/DDN-v6.pth


In [8]:
#@title (11) Load Pretrained model from Hugging Face
print("\n--- Testing Hugging Face Download & Load ---")
hf_model = DNN().to(DEVICE)
hf_model = load_model_hugging_face(new_model_c=hf_model, filename_c=FILENAME)
predicted_value = invoke_model_with_a_b_c(hf_model, 22, 1, 10)

MODEL = hf_model



--- Testing Hugging Face Download & Load ---


DDN-v6.pth: reconstructing file:   0%|          |  0.00B / 4.38MB            

DDN-v6.pth: downloading bytes:           |  0.00B            

model downloaded from HF to: /root/.cache/huggingface/hub/models--howardyang2009--llm-workshop-v1/snapshots/9c7582330429647fada5baf1f402411f1a934cfd/DDN-v6.pth
model weights loaded successfully into a new model instance.
Prediction for (22 * 1) % 10 = 2
